# Protein degree — biology, or bias?
### Fully reproducible pipeline

This notebook reproduces every figure in the deck from public data, with **no proprietary
connectors** — it queries the STRING and EBI Complex Portal REST APIs directly.

**The argument, in four steps:**
1. Build a real interaction network over 3 curated complexes + 8 abundant "hub" candidates.
2. Show the connectivity leaderboard ranks a hub (**ACTB**) #1 and a true member (**MCM6**) last.
3. **Prove degree is a confound, not biology** — a predictor using *only* node degree scores
   AUC ≈ 0.70 and keeps that accuracy after every real interaction is scrambled
   (degree-preserving null; the node-degree confound is in the spirit of
   Bernett et al. 2024, *Brief. Bioinform.* 25(2):bbae076 — a different method).
4. Recover real signal with a 3-step filter (specificity × evidence × membership).

Run top to bottom. Requires internet. Cells are seeded for determinism.


In [ ]:
# --- 0. Setup ---------------------------------------------------------------
import urllib.request, urllib.parse, json, itertools, sys
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib as mpl
from math import log

RNG = np.random.default_rng(0)   # global seed for reproducibility

# Optional: use Helvetica if present (falls back silently otherwise)
import matplotlib.font_manager as fm, os
for pth in ["/System/Library/Fonts/Helvetica.ttc"]:
    if os.path.exists(pth):
        fm.fontManager.addfont(pth); mpl.rcParams["font.family"]="Helvetica"
mpl.rcParams["axes.unicode_minus"]=False

# Deck palette
ARTIFACT="#E4572E"; REAL="#2A9D8F"; GREY_TXT="#5F6368"
print("Python", sys.version.split()[0], "| networkx", nx.__version__)

## 1. Inputs — the demo complexes and hub candidates
Three well-characterised human complexes, plus eight abundant/sticky proteins that are *not* members of these three (but are real, heavily-studied proteins).

In [ ]:
# --- 1. Fixed inputs --------------------------------------------------------
COMPLEXES = {
    "Complex A — Arp2/3":  ["ACTR2","ACTR3","ARPC1B","ARPC2","ARPC3","ARPC4","ARPC5"],
    "Complex B — CCT/TRiC": ["TCP1","CCT2","CCT3","CCT4","CCT5","CCT6A","CCT7","CCT8"],
    "Complex C — MCM":     ["MCM2","MCM3","MCM4","MCM5","MCM6","MCM7"],
}
HUB_CANDIDATES = ["ACTB","GAPDH","HSP90AA1","HSPA8","YWHAZ","UBC","VCP","EEF1A1"]

# Canonical Complex Portal accessions (for the membership gate in step 4)
CPX = {"Complex A — Arp2/3":"CPX-2579","Complex B — CCT/TRiC":"CPX-6030","Complex C — MCM":"CPX-2940"}

SYMBOLS = sorted(set(sum(COMPLEXES.values(), []) + HUB_CANDIDATES))
gene2cx = {g:c for c,ms in COMPLEXES.items() for g in ms}
REQUIRED_SCORE = 400   # STRING medium-confidence for building the network
HIGH_CONF      = 0.70  # high-confidence floor used by the evidence gate
print(len(SYMBOLS), "proteins:", SYMBOLS)

## 2. Build the network from the STRING REST API
One call to `string-db.org`. STRING returns each undirected edge once with a combined score in [0,1]. This reproduces the exact 178-edge network used in the deck.

In [ ]:
# --- 2. Fetch STRING network ------------------------------------------------
def fetch_string_network(symbols, species=9606, required_score=REQUIRED_SCORE):
    params = urllib.parse.urlencode({
        "identifiers": "%0d".join(symbols), "species": species,
        "required_score": required_score, "caller_identity": "degree_bias_repro"})
    url = "https://string-db.org/api/tsv/network?" + params
    with urllib.request.urlopen(url, timeout=60) as r:
        rows = r.read().decode().strip().split("\n")
    hdr = rows[0].split("\t")
    ia, ib, isc = hdr.index("preferredName_A"), hdr.index("preferredName_B"), hdr.index("score")
    G = nx.Graph(); G.add_nodes_from(symbols)
    for line in rows[1:]:
        f = line.split("\t")
        a, b, s = f[ia], f[ib], float(f[isc])
        if a in G and b in G:
            G.add_edge(a, b, score=s)
    return G

G = fetch_string_network(SYMBOLS)
print(f"nodes={G.number_of_nodes()}  edges={G.number_of_edges()}  "
      f"(deck used 29 nodes / 178 edges)")

## 3. The connectivity leaderboard
Score each protein by its **weighted degree** = sum of STRING confidences to its partners — the naive 'how connected is it' score. This is what puts a hub on top.

In [ ]:
# --- 3. Weighted-degree leaderboard ----------------------------------------
wdeg = {p: sum(G[p][q]["score"] for q in G.neighbors(p)) for p in G.nodes()}
board = sorted(wdeg, key=lambda p:-wdeg[p])
rank = {p:i+1 for i,p in enumerate(board)}
def role(p): return "hub (not in demo complexes)" if p not in gene2cx else gene2cx[p].split("—")[0].strip()
print(f"{'#':>3} {'protein':9s} {'score':>6}  role")
for p in board[:5] + ["..."] + board[-5:]:
    if p=="...": print("     ...."); continue
    print(f"{rank[p]:>3} {p:9s} {wdeg[p]:>6.2f}  {role(p)}")
print(f"\nACTB rank #{rank['ACTB']}  |  MCM6 rank #{rank['MCM6']}")

## 4. Proof: degree is a confound, not biology
The key test. We build a real **predicted probability** (`y_prob`) that a protein pair interacts, using proper held-out link prediction (test edges removed before features are computed — no leakage). Then:

- A **degree-only** model (preferential attachment) already scores AUC ≈ 0.70.
- Re-evaluating it on **degree-preserving scrambles** (all real partnerships randomised, every node's degree kept) barely changes the AUC.

If the accuracy survives destroying the biology, the skill was degree — not biology. (This targets the same node-degree confound Bernett et al. 2024 identified for sequence-based predictors; the method here — a Maslov–Sneppen edge-swap null — is our own, not their KaHIP/CD-HIT split control.)

In [ ]:
# --- 4a. AUC helper (Mann-Whitney) -----------------------------------------
def auc(labels, scores):
    labels = np.asarray(labels); scores = np.asarray(scores, float)
    uniq, inv, cnt = np.unique(scores, return_inverse=True, return_counts=True)
    csum = np.cumsum(cnt); avg = (csum - cnt + csum + 1) / 2.0
    r = avg[inv]
    P = labels.sum(); N = len(labels) - P
    if P == 0 or N == 0: return np.nan
    return (r[labels == 1].sum() - P*(P+1)/2) / (P*N)

ALLPAIRS = list(itertools.combinations(sorted(G.nodes()), 2))

def features(graph, pairs, feats):
    deg = dict(graph.degree())
    X = []
    for u, v in pairs:
        row = []
        if "PA"  in feats: row.append(np.log1p(deg[u]*deg[v]))
        if "SUM" in feats: row.append(deg[u]+deg[v])
        if "CN"  in feats: row.append(len(list(nx.common_neighbors(graph, u, v))))
        if "JAC" in feats:
            nu, nv = set(graph[u]), set(graph[v]); un = len(nu|nv)
            row.append(len(nu & nv)/un if un else 0.0)
        X.append(row)
    return np.array(X, float)

def fit_logreg(X, y, iters=300, lr=0.3):
    mu, sd = X.mean(0), X.std(0) + 1e-9
    Xs = np.c_[np.ones(len(X)), (X-mu)/sd]; w = np.zeros(Xs.shape[1])
    for _ in range(iters):
        p = 1/(1+np.exp(-Xs@w)); w -= lr * Xs.T@(p-y)/len(y)
    return w, mu, sd

def predict(w, mu, sd, X):
    Xs = np.c_[np.ones(len(X)), (X-mu)/sd]; return 1/(1+np.exp(-Xs@w))

def cv_auc(graph, feats, k=5, seed=0, return_roc=False):
    r = np.random.default_rng(seed); edges = list(graph.edges()); r.shuffle(edges)
    folds = np.array_split(np.arange(len(edges)), k)
    nonedges = [p for p in ALLPAIRS if not graph.has_edge(*p)]
    Y, P = [], []
    for f in folds:
        test_pos = [edges[i] for i in f]
        Gtr = graph.copy(); Gtr.remove_edges_from(test_pos)
        tr_pos = [e for i, e in enumerate(edges) if i not in set(f.tolist())]
        Xtr = features(Gtr, tr_pos+nonedges, feats)
        ytr = np.r_[np.ones(len(tr_pos)), np.zeros(len(nonedges))]
        w, mu, sd = fit_logreg(Xtr, ytr)
        Xte = features(Gtr, test_pos+nonedges, feats)
        yte = np.r_[np.ones(len(test_pos)), np.zeros(len(nonedges))]
        Y += list(yte); P += list(predict(w, mu, sd, Xte))
    return (auc(Y, P), np.array(Y), np.array(P)) if return_roc else auc(Y, P)

auc_deg  = np.mean([cv_auc(G, ["PA","SUM"], seed=s) for s in range(15)])
auc_full = np.mean([cv_auc(G, ["PA","SUM","CN","JAC"], seed=s) for s in range(15)])
print(f"degree-only y_prob : AUC = {auc_deg:.3f}")
print(f"full model  y_prob : AUC = {auc_full:.3f}")

In [ ]:
# --- 4b. Degree-preserving null (Maslov-Sneppen) ---------------------------
# NOTE: N_NULL controls runtime. 300 ~ under a minute; the deck used 300.
N_NULL = 300
m = G.number_of_edges()
null_auc = []
for it in range(N_NULL):
    H = G.copy()
    nx.double_edge_swap(H, nswap=10*m, max_tries=100*m, seed=int(RNG.integers(1e9)))
    null_auc.append(cv_auc(H, ["PA","SUM"], seed=it))
null_auc = np.array(null_auc)
print(f"real network AUC      = {auc_deg:.3f}")
print(f"scrambled network AUC = {np.median(null_auc):.3f} "
      f"(IQR {np.percentile(null_auc,25):.3f}-{np.percentile(null_auc,75):.3f})")
print("=> accuracy barely moves: the classifier's skill is DEGREE, not biology.")

In [ ]:
# --- 4c. Figure: y_prob ROC + degree-leakage null --------------------------
_, Y_do, P_do = cv_auc(G, ["PA","SUM"], seed=0, return_roc=True)
_, Y_fu, P_fu = cv_auc(G, ["PA","SUM","CN","JAC"], seed=0, return_roc=True)
def roc(y, p):
    o = np.argsort(-p); y = np.asarray(y)[o]
    P, N = y.sum(), len(y)-y.sum()
    return np.r_[0, np.cumsum(1-y)/N], np.r_[0, np.cumsum(y)/P]

BG="#0E1512"; PANEL="#141C18"; INK="#E8EDEA"; SUB="#9AA7A0"; GRID="#26302B"
A2="#E8654A"; R2="#37B89A"; G2="#6C7A72"
fig = plt.figure(figsize=(13.6,6.4)); fig.patch.set_facecolor(BG)
gs = fig.add_gridspec(1,2,left=.065,right=.975,top=.80,bottom=.13,wspace=.22)

axA = fig.add_subplot(gs[0,0]); axA.set_facecolor(PANEL)
for s in axA.spines.values(): s.set_color(GRID)
axA.tick_params(colors=SUB, labelsize=9)
axA.plot([0,1],[0,1],color=G2,lw=1,ls="--")
fx,ty = roc(Y_fu,P_fu); axA.plot(fx,ty,color=INK,lw=2.2,label=f"full model   AUC {auc_full:.2f}")
fx,ty = roc(Y_do,P_do); axA.plot(fx,ty,color=A2,lw=2.2,label=f"degree only  AUC {auc_deg:.2f}")
axA.set_xlabel("false-positive rate",color=INK,fontsize=10.5)
axA.set_ylabel("true-positive rate",color=INK,fontsize=10.5)
axA.set_title("A   A real y_prob — how much is just degree?",color=INK,fontsize=11,fontweight="bold",loc="left",pad=8)
axA.legend(loc="lower right",fontsize=9.5,frameon=False,labelcolor=INK); axA.set_xlim(0,1); axA.set_ylim(0,1.02)

axB = fig.add_subplot(gs[0,1]); axB.set_facecolor(PANEL)
for s in axB.spines.values(): s.set_color(GRID)
axB.tick_params(colors=SUB, labelsize=9)
axB.hist(null_auc,bins=20,color=G2,alpha=.85,label="biology-scrambled\n(degree preserved)")
axB.axvline(auc_deg,color=A2,lw=2.2,label=f"real network  AUC {auc_deg:.2f}")
axB.axvline(0.5,color=SUB,lw=1,ls=":")
axB.set_xlabel("degree-only classifier AUC",color=INK,fontsize=10.5)
axB.set_ylabel("count of randomizations",color=INK,fontsize=10.5)
axB.set_title("B   Destroy the biology, keep degree — AUC barely moves",color=INK,fontsize=11,fontweight="bold",loc="left",pad=8)
axB.legend(loc="upper left",fontsize=9,frameon=False,labelcolor=INK)
fig.text(.065,.95,"CAN A PPI PREDICTOR SUCCEED ON DEGREE ALONE?  YES.",color=A2,fontsize=11,fontweight="bold")
fig.text(.065,.885,f"A y_prob using only partner counts scores AUC {auc_deg:.2f} and holds it after all real biology is randomized away.",color=INK,fontsize=12)
fig.text(.065,.03,"STRING v12 · 5-fold held-out link prediction · logistic y_prob · degree-preserving edge-swap null (node-degree confound in the spirit of Bernett et al. 2024; different method)",color=G2,fontsize=8,style="italic")
fig.savefig("fig_degree_leakage.png",dpi=200,facecolor=BG,bbox_inches="tight"); plt.show()

## 5. Recovering real signal — the 3-step filter
Degree is a confound, but the network still contains real biology. To turn a PPI *hit* into a candidate *target* we apply three filters in order:

1. **Specificity weight** (TF-IDF style): down-weight frequent flyers that connect into many complexes.
2. **Evidence gate**: only high-confidence edges (STRING ≥ 0.70) count.
3. **Membership gate**: require curated Complex Portal membership — the step that separates a genuine *interactor* (e.g. HSPA8↔CCT) from an actual *member*.

In [ ]:
# --- 5a. Curated membership from Complex Portal REST ------------------------
def fetch_members(cpx_ac):
    url = f"https://www.ebi.ac.uk/intact/complex-ws/complex/{cpx_ac}"
    req = urllib.request.Request(url, headers={"Accept":"application/json"})
    with urllib.request.urlopen(req, timeout=60) as r:
        c = json.load(r)
    return sorted({p.get("name") for p in c.get("participants", [])})

curated = {name: set(fetch_members(ac)) for name, ac in CPX.items()}
for name, s in curated.items():
    print(f"{name:22s} [{CPX[name]}]: {sorted(s)}")

In [ ]:
# --- 5b. Three-step re-ranking ---------------------------------------------
CX = list(COMPLEXES); NC = len(CX)
def n_cx_touched(p):
    return len({gene2cx[nb] for nb in G.neighbors(p)
                if G[p][nb]["score"] >= HIGH_CONF and nb in gene2cx})
idf = {p: log(NC/(1+n_cx_touched(p))) + 1e-9 for p in G.nodes()}   # step 1

def naive_score(p, C):      # step 0
    return sum(G[p][m]["score"] for m in COMPLEXES[C] if m != p and G.has_edge(p, m))
def corrected_score(p, C):  # steps 1 x 2
    ev = sum(G[p][m]["score"] for m in COMPLEXES[C]
             if m != p and G.has_edge(p, m) and G[p][m]["score"] >= HIGH_CONF)
    return ev * idf[p]

def rank_by(fn, C):
    order = sorted(G.nodes(), key=lambda p:-fn(p, C))
    return {p:i+1 for i, p in enumerate(order)}

def verdict(p, C):          # step 3 promotion logic
    member = p in curated[C]; top = rank_by(corrected_score, C)[p] <= 8
    if member and top: return "confirmed target"
    if member:         return "member, weak network signal"
    if top:            return "hypothesis only (interactor, not member)"
    return "deprioritized"

for C in CX:
    nv, cr = rank_by(naive_score, C), rank_by(corrected_score, C)
    print(f"\n=== {C.split('—')[0].strip()} : naive#1 -> corrected#1 ===")
    for p in sorted(G.nodes(), key=lambda x: nv[x])[:4]:
        print(f"  naive#{nv[p]:<2} {p:8s} corr#{cr[p]:<2}  {verdict(p, C)}")

### What the filter fixes
- **ACTB** (naive #1 in Complex A) → *hypothesis only (interactor, not member)*.
- **HSPA8** (naive #1 in Complex B) → *hypothesis only* — it genuinely binds all 8 CCT subunits (HSP70/chaperonin cooperation), so only the **membership gate** can demote it. This is why connectivity alone, even *specific* connectivity, cannot define a target.
- Every true subunit is retained as a **confirmed target**.

**Bottom line:** PPI gives a *hypothesis*, not a target. Rank by connectivity and you validate hubs; add specificity + evidence + membership and you recover the real members.

---
#### Reproducibility notes
- Data: STRING v12 (`string-db.org/api`) and EBI Complex Portal (`ebi.ac.uk/intact/complex-ws`), fetched live — re-running reflects any database updates.
- Determinism: all randomness seeded (`RNG`, per-fold seeds). `N_NULL` trades runtime for smoothness.
- The membership gate uses curated truth (valid for these textbook complexes). For a novel complex, replace step 3 with orthogonal functional evidence (co-essentiality, co-expression, structure).

In [ ]:
# --- 6. Environment stamp --------------------------------------------------
import platform
print("python  :", platform.python_version())
for mod in ("numpy","networkx","matplotlib"):
    print(f"{mod:11s}:", __import__(mod).__version__)
print("\nAll figures written: fig_degree_leakage.png")